# Notebook 4 — Structural and Functional Implications

## Learning objectives
- Map convergent alignment positions to **sequence positions** in a reference
- Classify convergent residues by **protein domain** (TMD vs STAS)
- Generate a **PyMOL script** to visualize convergent sites on the 3D structure
- Assess whether convergence targets **functionally important regions**
- Synthesize all evidence: phylogeny + alignment + structure

In [ ]:
import os, re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from Bio import AlignIO, SeqIO
from scipy.stats import fisher_exact

DATA = os.path.join("..", "data")
SUB_DIR = os.path.join(DATA, "subfamilies")
FIGS = os.path.join("..", "figures")
AA = set("ACDEFGHIKLMNPQRSTVWY")

---
## 1. Reload convergent sites from Notebook 3

Re-run the detection (or import saved results).

In [ ]:
# Copy ECHOLOCATING_TAXIDS from Notebook 3
ECHOLOCATING_TAXIDS = {
    "9739", "9749", "9755",
    # ... your bat taxids
}

# Load prestin alignment (same as Notebook 3)
prestin_alns = sorted([f for f in os.listdir(SUB_DIR)
                       if "prestin" in f and "gt" in f])
PRESTIN_ALN = os.path.join(SUB_DIR, prestin_alns[0]) if prestin_alns else None

aln = AlignIO.read(PRESTIN_ALN, "fasta")
echo_idx = [i for i in range(len(aln)) if aln[i].id.split(".")[0] in ECHOLOCATING_TAXIDS]
nonecho_idx = [i for i in range(len(aln)) if i not in echo_idx]

# Re-detect convergent sites
def find_convergent_sites(aln, echo_idx, nonecho_idx, min_echo_frac=0.6, min_echo_count=3):
    sites = []
    for pos in range(aln.get_alignment_length()):
        echo_aas = [str(aln[i].seq[pos]) for i in echo_idx if str(aln[i].seq[pos]) in AA]
        nonecho_aas = [str(aln[i].seq[pos]) for i in nonecho_idx if str(aln[i].seq[pos]) in AA]
        if len(echo_aas) < min_echo_count or len(nonecho_aas) < 3: continue
        echo_counts = Counter(echo_aas)
        top_aa, top_count = echo_counts.most_common(1)[0]
        echo_frac = top_count / len(echo_aas)
        if echo_frac < min_echo_frac: continue
        nonecho_top = Counter(nonecho_aas).most_common(1)[0][0]
        if top_aa != nonecho_top:
            nonecho_has = Counter(nonecho_aas).get(top_aa, 0) / len(nonecho_aas)
            sites.append({"position": pos, "echo_aa": top_aa, "echo_fraction": echo_frac,
                          "nonecho_consensus": nonecho_top, "specificity": echo_frac - nonecho_has})
    return sites

conv_sites = find_convergent_sites(aln, echo_idx, nonecho_idx)
print(f"Convergent sites: {len(conv_sites)}")

---
## 2. Mapping alignment positions → sequence positions

The alignment has gaps. To map onto the 3D structure, we need the
**ungapped position** in a reference sequence.

We use human prestin. The human taxid is 9606.

In [ ]:
# Find the human sequence
human_idx = None
for i, rec in enumerate(aln):
    if rec.id.startswith("9606."):
        human_idx = i
        break

if human_idx is None:
    print("⚠ Human sequence not found. Using first sequence as reference.")
    human_idx = 0

ref = aln[human_idx]
print(f"Reference: {ref.id}")
print(f"Aligned length: {len(ref.seq)}")
ungapped = str(ref.seq).replace("-", "")
print(f"Ungapped length: {len(ungapped)}")

In [ ]:
# Build alignment → sequence position mapping (1-based)
def aln_to_seq_map(aligned_seq):
    """Map alignment column (0-based) → sequence position (1-based)."""
    mapping = {}
    seq_pos = 0
    for aln_pos, aa in enumerate(str(aligned_seq)):
        if aa != "-":
            seq_pos += 1
            mapping[aln_pos] = seq_pos
    return mapping

pos_map = aln_to_seq_map(ref.seq)

# Map convergent sites
mapped = []
for s in conv_sites:
    sp = pos_map.get(s["position"])
    if sp is not None:
        mapped.append({**s, "seq_pos": sp})

print(f"\nMapped {len(mapped)} / {len(conv_sites)} convergent sites to reference sequence")
print(f"\n{'AlnPos':>7s}  {'SeqPos':>7s}  {'EchoAA':>7s}  {'NonEcho':>7s}  {'Specif':>7s}")
print("-" * 42)
for s in sorted(mapped, key=lambda x: x["seq_pos"]):
    print(f"{s['position']:>7d}  {s['seq_pos']:>7d}  {s['echo_aa']:>7s}  "
          f"{s['nonecho_consensus']:>7s}  {s['specificity']:>7.3f}")

---
## 3. Domain mapping

Human prestin (UniProt Q7LBE3, ~744 residues):

| Region | Residues | Function |
|:---|:---|:---|
| N-terminal | 1–80 | Cytoplasmic, regulatory |
| **TMD** | **81–505** | **Transmembrane motor domain (14 TM helices)** |
| Linker | 506–530 | Connects TMD to STAS |
| **STAS** | **531–744** | **C-terminal regulatory domain** |

Adjust boundaries if your reference sequence differs.

In [ ]:
DOMAINS = {
    "N-term":  (1, 80),
    "TMD":     (81, 505),
    "Linker":  (506, 530),
    "STAS":    (531, 744),
}

domain_counts = Counter()
for s in mapped:
    for name, (start, end) in DOMAINS.items():
        if start <= s["seq_pos"] <= end:
            s["domain"] = name
            domain_counts[name] += 1
            break
    else:
        s["domain"] = "outside"
        domain_counts["outside"] += 1

total_mapped = len(mapped)
print(f"{'Domain':<10s} {'Sites':>6s} {'Expected':>9s} {'Enrich':>8s}")
print("-" * 38)
for name, (start, end) in DOMAINS.items():
    n = domain_counts[name]
    dlen = end - start + 1
    expected = total_mapped * dlen / len(ungapped) if total_mapped > 0 else 0
    enrichment = n / expected if expected > 0 else 0
    print(f"  {name:<10s} {n:>5d}  {expected:>8.1f}  {enrichment:>7.1f}×")

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(14, 3))
domain_colors = {"N-term": "#CCCCCC", "TMD": "#D85A30", "Linker": "#EF9F27", "STAS": "#85B7EB"}
for name, (start, end) in DOMAINS.items():
    ax.barh(0, end-start, left=start, height=0.6, color=domain_colors[name],
            alpha=0.7, edgecolor="white", label=name)
for s in mapped:
    ax.plot(s["seq_pos"], 0, "|", color="black", markersize=15, markeredgewidth=1.5)
ax.set_xlim(0, len(ungapped) + 10)
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_xlabel("Residue position (human prestin)")
ax.set_title(f"Convergent residues on prestin domains ({len(mapped)} sites)", fontweight="bold")
ax.legend(loc="upper right", ncol=4)
plt.tight_layout()
plt.savefig(os.path.join(FIGS, "04_domain_map.png"), dpi=150, bbox_inches="tight")
plt.show()

### ✏️ Exercise 4.1
Is the TMD **enriched** for convergent sites? Use Fisher's exact test:

In [ ]:
tmd_len = DOMAINS["TMD"][1] - DOMAINS["TMD"][0] + 1
stas_len = DOMAINS["STAS"][1] - DOMAINS["STAS"][0] + 1
tmd_conv = domain_counts.get("TMD", 0)
stas_conv = domain_counts.get("STAS", 0)

table = [[tmd_conv, tmd_len - tmd_conv],
         [stas_conv, stas_len - stas_conv]]
odds, p = fisher_exact(table, alternative="greater")
print(f"TMD:  {tmd_conv} convergent / {tmd_len} residues")
print(f"STAS: {stas_conv} convergent / {stas_len} residues")
print(f"Fisher's exact: OR = {odds:.2f}, p = {p:.4f}")

---
## 4. PyMOL visualization script

Prestin structure: PDB **7S8X** (Bavi et al. 2021, Nature 600, 553–558).

In [ ]:
PYMOL_SCRIPT = os.path.join(FIGS, "prestin_convergent.pml")
residue_list = "+".join(str(s["seq_pos"]) for s in mapped)

script = f"""# Prestin convergent residues — PyMOL script
# Usage: pymol {os.path.basename(PYMOL_SCRIPT)}

fetch 7S8X, prestin
remove solvent
remove chain B

hide everything
show cartoon, prestin
set cartoon_transparency, 0.3

# Color by domain
color gray80, prestin
color tv_red, prestin and resi {DOMAINS['TMD'][0]}-{DOMAINS['TMD'][1]}
color tv_orange, prestin and resi {DOMAINS['Linker'][0]}-{DOMAINS['Linker'][1]}
color tv_blue, prestin and resi {DOMAINS['STAS'][0]}-{DOMAINS['STAS'][1]}

# Convergent residues as yellow spheres
select convergent, prestin and resi {residue_list}
show spheres, convergent and name CA
color yellow, convergent and name CA
set sphere_scale, 0.8, convergent

# Labels
"""
for s in mapped:
    script += f'label prestin and resi {s["seq_pos"]} and name CA, "{s["echo_aa"]}{s["seq_pos"]}"\n'

script += """
set label_size, 12
set label_color, black
bg_color white
orient prestin

# Render:
# ray 2400, 1800
# png prestin_convergent.png, dpi=300
"""

with open(PYMOL_SCRIPT, "w") as f:
    f.write(script)

print(f"PyMOL script: {PYMOL_SCRIPT}")
print(f"Run: pymol {PYMOL_SCRIPT}")
print(f"\nYellow spheres = convergent residues on the red (TMD) / blue (STAS) structure")

### ✏️ Exercise 4.2
Open the structure in PyMOL:
1. Are convergent residues **surface-exposed** or **buried**?
2. Do they cluster near the **anion binding site** (central TM cavity)?
3. Are they on the same face of the protein?

To check proximity to the binding site:
```
select anion_site, prestin and resi 137+396
distance d1, convergent, anion_site
```

### ✏️ Exercise 4.3
Modify the PyMOL script to color convergent residues differently
based on their **specificity** score (high specificity = bright red,
low = pale orange). Use PyMOL's `set_color` and `color` commands.

---
## 5. Synthesis of evidence

In [ ]:
print("=" * 65)
print("EVIDENCE FOR CONVERGENT MOLECULAR EVOLUTION IN PRESTIN")
print("=" * 65)
print(f"""
1. PHYLOGENETIC SIGNAL (Notebooks 1–2)
   Prestin trees group echolocating species together,
   especially with distance-based methods (NJ).
   Control subfamilies recover the species tree.

2. CONVERGENT RESIDUES (Notebook 3)
   {len(conv_sites)} convergent sites detected in prestin.
   Permutation test confirms this exceeds chance expectation.

3. STRUCTURAL LOCALIZATION (this notebook)
   {domain_counts.get('TMD', 0)} convergent sites in the TMD (motor domain)
   {domain_counts.get('STAS', 0)} in the STAS domain
   TMD enrichment Fisher's p = {p:.4f}

4. INTERPRETATION
   Independent evolution of echolocation in bats and dolphins
   drove convergent amino acid substitutions in prestin,
   concentrated in the transmembrane motor domain where
   voltage-dependent electromotility occurs.
""")

---
## 6. Discussion

### ✏️ Exercise 4.4
1. Could convergence result from **relaxation of constraint** rather than
   positive selection? How would dN/dS analysis help distinguish these?
2. How does our **permutation test** address the concerns of
   Thomas & Hahn (2015) about false positives in convergence detection?
3. What experiment would **prove** that a specific convergent residue
   affects prestin's motor function? (Think: mutagenesis + electrophysiology)
4. Fruit bats (Pteropodidae) lost echolocation secondarily. What would
   you predict about their prestin sequence at convergent positions?
5. SLC26A4 (pendrin) is also hearing-related. Would you expect weaker
   convergent signal there? How would you test it?

## References
- Parker et al. (2013) Nature 502, 228–231
- Bavi et al. (2021) Nature 600, 553–558 (PDB 7S8X)
- Li et al. (2008) PNAS 105, 13959–13964
- Thomas & Hahn (2015) Mol Biol Evol 32, 1232–1236